# TabPFN with LSOA commute features (Swindon)

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

DATA_CANDIDATES = [
    Path("../commuting-regression/data_swindon_with_lsoa_commute.csv"),
    Path("commuting-regression/data_swindon_with_lsoa_commute.csv"),
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not locate data_swindon_with_lsoa_commute.csv")
TARGET = "log_total_GVA_2023"
ID_COLUMNS = ["LSOA21CD", "MSOA21CD"]
METADATA_COLUMNS = ID_COLUMNS + ["is_swindon", TARGET]

df = pd.read_csv(DATA_PATH)

assert df["LSOA21CD"].is_unique, "LSOA21CD must be unique"
assert df[TARGET].notna().all(), "Target contains missing values"
assert not any(c.startswith("msoa_") for c in df.columns), (
    "Legacy MSOA commute features are still present"
)

print(f"Dataset: {DATA_PATH.resolve()}")
print(f"Rows: {len(df)} | LSOAs: {df['LSOA21CD'].nunique()} | MSOAs: {df['MSOA21CD'].nunique()}")

Dataset: D:\github\G23-Swindon-Borough-Council\commuting-regression\data_swindon_with_lsoa_commute.csv
Rows: 137 | LSOAs: 137 | MSOAs: 27


In [2]:
commute_features = [c for c in df.columns if c.startswith("lsoa_")]
engineered_features = [
    c for c in df.columns
    if c not in METADATA_COLUMNS and not c.startswith("lsoa_")
]
all_features = engineered_features + commute_features

non_numeric = df[all_features].select_dtypes(exclude=np.number).columns.tolist()
assert not non_numeric, f"Non-numeric model features: {non_numeric}"
assert not df[all_features].isna().any().any(), "Model features contain missing values"
assert np.isfinite(df[all_features].to_numpy()).all(), "Model features contain infinite values"

feature_sets = {
    "engineered only": engineered_features,
    "engineered + LSOA commute": all_features,
}

feature_audit = pd.DataFrame({
    "feature_set": feature_sets.keys(),
    "n_features": [len(v) for v in feature_sets.values()],
})
display(feature_audit)
print("LSOA commute features:")
print(f"- {c}" for c in commute_features)

,feature_set,n_features
0,engineered only,10
1,engineered + LSOA commute,25



LSOA commute features:
- lsoa_total_employed
- lsoa_home_or_no_fixed_count
- lsoa_home_or_no_fixed_share
- lsoa_workplace_commuters
- lsoa_same_lsoa_worker_count
- lsoa_same_lsoa_work_share
- lsoa_out_commute_share
- lsoa_workers_at_workplace
- lsoa_inbound_worker_count
- lsoa_local_worker_share
- lsoa_in_commute_share
- lsoa_outbound_from_swindon_count
- lsoa_outbound_from_swindon_share
- lsoa_inbound_to_swindon_count
- lsoa_inbound_to_swindon_share


In [3]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import GroupKFold
from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

N_SPLITS = 5
groups = df["MSOA21CD"].to_numpy()
y = df[TARGET].to_numpy(dtype=float)

In [4]:
def grouped_tabpfn_cv(feature_names, n_splits=N_SPLITS):
    X = df[feature_names].to_numpy(dtype=float)
    splitter = GroupKFold(n_splits=n_splits)
    oof = np.full(len(df), np.nan, dtype=float)
    fold_id = np.full(len(df), -1, dtype=int)

    for fold, (train_idx, valid_idx) in enumerate(
        splitter.split(X, y, groups), start=1
    ):
        model = TabPFNRegressor.create_default_for_version(ModelVersion.V3)
        model.fit(X[train_idx], y[train_idx])
        oof[valid_idx] = np.asarray(model.predict(X[valid_idx])).reshape(-1)
        fold_id[valid_idx] = fold
        print(
            f"fold {fold}/{n_splits}: train={len(train_idx)}, "
            f"validation={len(valid_idx)}, validation MSOAs={len(np.unique(groups[valid_idx]))}"
        )

    assert np.isfinite(oof).all()
    return oof, fold_id

In [5]:
metric_rows = []
oof_predictions = {}
fold_membership = None

for feature_set_name, feature_names in feature_sets.items():
    print(f"\nRunning: {feature_set_name} ({len(feature_names)} features)")
    oof, folds = grouped_tabpfn_cv(feature_names)
    oof_predictions[feature_set_name] = oof
    if fold_membership is None:
        fold_membership = folds
    else:
        assert np.array_equal(fold_membership, folds)

    metric_rows.append({
        "feature_set": feature_set_name,
        "n_features": len(feature_names),
        "MAE": mean_absolute_error(y, oof),
        "RMSE": mean_squared_error(y, oof) ** 0.5,
        "MAPE": mean_absolute_percentage_error(y, oof),
        "R2": r2_score(y, oof),
    })

results = pd.DataFrame(metric_rows).sort_values("R2", ascending=False).reset_index(drop=True)
display(results.style.format({"MAE": "{:.4f}", "RMSE": "{:.4f}", "MAPE": "{:.2%}", "R2": "{:.4f}"}))


Running: engineered only (10 features)
fold 1/5: train=107, validation=30, validation MSOAs=6
fold 2/5: train=108, validation=29, validation MSOAs=6
fold 3/5: train=111, validation=26, validation MSOAs=5
fold 4/5: train=111, validation=26, validation MSOAs=5
fold 5/5: train=111, validation=26, validation MSOAs=5

Running: engineered + LSOA commute (25 features)
fold 1/5: train=107, validation=30, validation MSOAs=6
fold 2/5: train=108, validation=29, validation MSOAs=6
fold 3/5: train=111, validation=26, validation MSOAs=5
fold 4/5: train=111, validation=26, validation MSOAs=5
fold 5/5: train=111, validation=26, validation MSOAs=5


,feature_set,n_features,MAE,RMSE,MAPE,R2
0,engineered only,10,0.4715,0.7523,13.13%,0.6595
1,engineered + LSOA commute,25,0.4872,0.7781,13.35%,0.6357


In [6]:
baseline_r2 = results.loc[results["feature_set"].eq("engineered only"), "R2"].iloc[0]
commute_r2 = results.loc[results["feature_set"].eq("engineered + LSOA commute"), "R2"].iloc[0]
print(f"R2 change after adding LSOA commute features: {commute_r2 - baseline_r2:+.4f}")

oof_results = df[ID_COLUMNS].copy()
oof_results["fold"] = fold_membership
oof_results["y_true"] = y
for name, values in oof_predictions.items():
    safe_name = name.lower().replace(" + ", "_plus_").replace(" ", "_")
    oof_results[f"y_pred_{safe_name}"] = values

display(oof_results.head(10))

R2 change after adding LSOA commute features: -0.0238


,LSOA21CD,MSOA21CD,fold,y_true,y_pred_engineered_only,y_pred_engineered_plus_lsoa_commute
0,E01015471,E02006849,3,2.776581,2.426692,2.210447
1,E01015473,E02003219,4,6.207100,6.505128,6.513515
2,E01015475,E02003226,3,4.612086,4.892155,3.489372
3,E01015477,E02003226,3,3.035241,3.099305,3.516100
4,E01015478,E02003232,5,5.274071,3.508306,3.591117
5,E01015479,E02003228,2,4.859223,4.896455,4.814967
6,E01015480,E02003226,3,2.652044,2.756974,2.715088
7,E01015481,E02003224,1,2.824410,2.794286,2.697465
8,E01015482,E02003224,1,4.366558,2.379006,2.912622
9,E01015483,E02003224,1,2.110213,2.193653,2.041248
